# Vectors, Matrices & Operations

### Learning objectives 

 1. Build a Matrix class with element-wise operaitons, matrix multiplication, transpose, determinant, and inverse
 2. Distinguise element-wise multiplication from matrix multiplicaiton and explain when each applies
 3. Implement a single dense neural network layer(```relu(W @ X + b) ```) using only the from scratch Matrix class
 4. Explain broadcasting rules and how bias addition works in neural network frameworks 

In [3]:
class Vector:
    def __init__(self,data):
        self.data = list(data)
        self.size = len(self.data)

    def __repr__(self):
        return f"Vector({self.data})"

    def __add__(self,other):
        return Vector([ a+b for a,b in zip(self.data,other.data)])

    def __sub__(self,other):
        return Vector([ a - b for a,b in zip(self.data,other.data)])
    def __mul__(self,scalar):
        return Vector([x * scalar for x in self.data])
    def dot(self,other):
        return sum(a * b for a,b in zip(self.data, other.data))

    def magnitude(self):
        return sum(x ** 2 for x in self.data) ** 0.5
    

In [8]:
class Matrix:
    def __init__(self,data):
        self.data = [list(row) for row in data]
        self.rows = len(self.data)
        self.cols = len(self.data[0])
        self.shape = (self.rows,self.cols)

    def __repr__(self):
        rows_str = "\n ".join(str(row) for row in self.data)
        return f"Matrix({self.shape}):\n {rows_str}"
    def __add__(self,other):
        return Matrix([
            [self.data[i][j] + other.data[i][j] for j in range(self.cols)]
            for i in range(self.rows)])
    def __sub__(self,other):
        return Matrix([
            [self.data[i][j] - other.data[i][j] for j in range(self.cols)]
            for i in range(self.rows)])
    def scalar_multiply(self, scalar):
        return Matrix([
            [self.data[i][j] * scalar for j in range(self.cols)]
            for i in range(self.rows)])
    def element_wise_multiply(self, other):
        return Matrix([
            [self.data[i][j] * other.data[i][j] for j in range(self.cols)]
            for i in range(self.rows)])
    def matmul(self,other):
        return Matrix([
            [
                sum(self.data[i][k] * other.data[i][j] for k in range(self.cols))
                for j in range(other.cols)
            ]
            for i in range(self.rows)
        ])
    def transpose(self):
        return Matrix([
            [self.data[j][i] for j in range(self.rows)]
            for i in range(self.cols)])
    def determinant(self):
        if self.shape == (1,1):
            return self.data[0][0]
        if self.shape == (2,2):
            return self.data[0][0] * self.data[1][1] - self.data[0][1] * self.data[1][0]
        det = 0
        for j in range(self.cols):
            minor = Matrix([
                [self.data[i][k] for k in range(self.cols) if k!=j]
                for i in range(1,self.rows)
            ])
            det  += (-1 ** j) * self.data[0][j] * minor.determinant()
        return det
    def inverse_2x2(self):
        det = self.determinant()
        if det == 0:
            raise ValueError("Matrix is singular, no inverse exists")
        return Matrix([
        [self.data[1][1] / det, -self.data[0][1] / det],
        [-self.data[1][0] / det, self.data[0][0] / det]
        ])
    @staticmethod
    def identity(n):
        return Matrix([
            [1 if i == j else 0 for j in range(n)]
            for i in range(n)])
        

In [9]:
A = Matrix([[1, 2], [3, 4]])
B = Matrix([[5, 6], [7, 8]])

print("A + B =", (A + B).data)
print("A @ B =", A.matmul(B).data)
print("A^T =", A.transpose().data)
print("det(A) =", A.determinant())
print("A^-1 =", A.inverse_2x2().data)

I = Matrix.identity(2)
print("A @ A^-1 =", A.matmul(A.inverse_2x2()).data)

A + B = [[6, 8], [10, 12]]
A @ B = [[15, 18], [49, 56]]
A^T = [[1, 3], [2, 4]]
det(A) = -2
A^-1 = [[-2.0, 1.0], [1.5, -0.5]]
A @ A^-1 = [[-6.0, 3.0], [10.5, -3.5]]


In [11]:
import random 

inputs = Matrix([[0.5] , [0.8] , [0.2]])
weights = Matrix([
    [random.uniform(-1,1) for _ in range(3)]
    for _ in range(2)])
bias = Matrix([[0.1], [0.1]])

def relu_matrix(m):
    return Matrix([[max(0,val) for val in row] for row in m.data])
pre_activation = weights.matmul(inputs) + bias
output = relu_matrix(pre_activation)

print(f"Input shape: {inputs.shape}")
print(f"Weight shape: {weights.shape}")
print(f"Output shape: {output.shape}")
print(f"Output: {output.data}")

Input shape: (3, 1)
Weight shape: (2, 3)
Output shape: (2, 1)
Output: [[0.3287334097148782], [0]]


## Now with numpy 

In [13]:
import numpy as np 
A = np.array([[1,2],[3,4]])
B = np.array([[5,6],[7,8]])

print("A + B =\n", A + B)
print("A * B (element-wise) =\n", A * B)
print("A @ B (matrix multiply) =\n", A @ B)
print("A^T =\n", A.T)
print("det(A) =", np.linalg.det(A))
print("A^-1 =\n", np.linalg.inv(A))
print("I = \n", np.eye(2))

inputs = np.random.randn(3,1)
weights = np.random.randn(2,3)
bias = np.array([[0.1], [0.1]])
output = np.maximum(0, weights @ inputs + bias)

print(f"\nNeural network layer: {weights.shape} @ {inputs.shape} = {output.shape}")
print(f"Ouput: \n{output}")


A + B =
 [[ 6  8]
 [10 12]]
A * B (element-wise) =
 [[ 5 12]
 [21 32]]
A @ B (matrix multiply) =
 [[19 22]
 [43 50]]
A^T =
 [[1 3]
 [2 4]]
det(A) = -2.0000000000000004
A^-1 =
 [[-2.   1. ]
 [ 1.5 -0.5]]
I = 
 [[1. 0.]
 [0. 1.]]

Neural network layer: (2, 3) @ (3, 1) = (2, 1)
Ouput: 
[[1.21329866]
 [0.        ]]


## Broadcasting with Numpy 
 **Note**: Numpy automatically broadcasts the 1D bias across both rows. This is how bias addition works in every neural network framework 

In [15]:
matrix = np.array([[1,2,3] , [4,5,6]])
bias = np.array([10,20,30])
print(matrix + bias)

[[11 22 33]
 [14 25 36]]
